In [0]:

%run ./blob_pipeline_lib

In [0]:
dbutils.widgets.text("STATE_SCHEMA", "4_prod.tmp")
dbutils.widgets.dropdown(
    "ACTION",
    "status",
    [
        "status",
        "quarantine",
        "requeue_key",
        "waive_key",
        "retry_failed_shards",
        "release_stale_leases",
        "gc_preview",
        "gc_execute",
    ],
)
dbutils.widgets.text("RUN_ID", "")
dbutils.widgets.text("EVENT_ID", "")
dbutils.widgets.text("ADC_UPDT", "")
dbutils.widgets.text("REVIEWER", "")
dbutils.widgets.text("RESOLUTION", "")
dbutils.widgets.text("GC_INBOX_RETENTION_DAYS", "7")
dbutils.widgets.text("GC_RUN_RETENTION_DAYS", "30")
dbutils.widgets.dropdown("DRY_RUN", "true", ["true", "false"])

In [0]:


STATE_SCHEMA = dbutils.widgets.get("STATE_SCHEMA").strip()
ACTION = dbutils.widgets.get("ACTION").strip()
RUN_ID = dbutils.widgets.get("RUN_ID").strip()
EVENT_ID_RAW = dbutils.widgets.get("EVENT_ID").strip()
ADC_UPDT_RAW = dbutils.widgets.get("ADC_UPDT").strip()
REVIEWER = dbutils.widgets.get("REVIEWER").strip()
RESOLUTION = dbutils.widgets.get("RESOLUTION").strip()
GC_INBOX_RETENTION_DAYS = int(dbutils.widgets.get("GC_INBOX_RETENTION_DAYS") or "7")
GC_RUN_RETENTION_DAYS = int(dbutils.widgets.get("GC_RUN_RETENTION_DAYS") or "30")
DRY_RUN = dbutils.widgets.get("DRY_RUN").lower() == "true"

validate_table_name(STATE_SCHEMA)

if GC_INBOX_RETENTION_DAYS < 7:
    raise ValueError("GC_INBOX_RETENTION_DAYS cannot be less than the 7-day Delta retention")
if GC_RUN_RETENTION_DAYS < 7:
    raise ValueError("GC_RUN_RETENTION_DAYS cannot be less than 7")

def state_table(name):
    value = f"{STATE_SCHEMA}.{name}"
    validate_table_name(value)
    return value

RUNS = state_table("pipeline_runs")
CHECKPOINT = state_table("pipeline_checkpoint")
COMMITS = state_table("source_commits")
INBOX = state_table("cdf_inbox")
RUN_EVENTS = state_table("run_events")
RUN_CHUNKS = state_table("run_chunks")
ATTEMPTS = state_table("shard_attempts")
BATCH = state_table("batch_output")
HISTORY = state_table("history_output")
QUARANTINE = state_table("quarantine")
FILE_QUEUE = state_table("file_queue")
EVENT_LOG = state_table("pipeline_events")

for required in (
    RUNS, CHECKPOINT, COMMITS, INBOX, RUN_EVENTS, RUN_CHUNKS,
    ATTEMPTS, BATCH, HISTORY, QUARANTINE, FILE_QUEUE, EVENT_LOG,
):
    if not table_exists(required):
        raise RuntimeError(f"Missing Blob v4 state table: {required}")

In [0]:
def display_status():
    print("Runs")
    display(
        spark.table(RUNS)
        .withColumn(
            "lease_is_stale",
            F.col("lease_owner").isNotNull()
            & F.col("lease_expires_ts").isNotNull()
            & (F.col("lease_expires_ts") < F.current_timestamp()),
        )
        .orderBy(F.col("updated_ts").desc_nulls_last())
        .limit(200)
    )

    print("Source commit/gap status")
    display(
        spark.table(COMMITS)
        .filter(
            F.col("ingest_status").isin(
                "cdf_gap_blocked", "cdf_unreadable", "pending", "spooled"
            )
            | (F.col("relevance_class") == "unknown")
        )
        .orderBy(F.col("version").desc())
        .limit(200)
    )

    print("Quarantine status")
    display(
        spark.table(QUARANTINE)
        .groupBy("status", "reason_code")
        .agg(
            F.count("*").alias("rows"),
            F.countDistinct("run_id").alias("runs"),
            F.max("last_failure_ts").alias("latest_failure_ts"),
        )
        .orderBy(F.desc("rows"))
    )

    print("Failed or stale shard attempts")
    display(
        spark.table(ATTEMPTS)
        .filter(
            (F.col("status").isin("failed", "lease_expired"))
            | (
                F.col("lease_expires_ts").isNotNull()
                & (F.col("lease_expires_ts") < F.current_timestamp())
            )
        )
        .orderBy(F.col("started_ts").desc_nulls_last())
        .limit(200)
    )

    print("File queue status")
    display(
        spark.table(FILE_QUEUE)
        .groupBy("status")
        .agg(
            F.count("*").alias("rows"),
            F.min("created_ts").alias("oldest_created_ts"),
            F.max("updated_ts").alias("latest_updated_ts"),
        )
        .orderBy(F.desc("rows"))
    )

    print("Recent pipeline events")
    display(
        spark.table(EVENT_LOG)
        .orderBy(F.col("event_ts").desc())
        .limit(200)
    )


def parsed_event_key():
    if not RUN_ID:
        raise ValueError("RUN_ID is required")
    if not EVENT_ID_RAW:
        raise ValueError("EVENT_ID is required")
    event_id = int(EVENT_ID_RAW)
    if ADC_UPDT_RAW:
        adc_updt = spark.sql(
            f"SELECT try_cast({sql_string(ADC_UPDT_RAW)} AS TIMESTAMP) AS value"
        ).first()["value"]
        if adc_updt is None:
            raise ValueError(f"ADC_UPDT is not a valid timestamp: {ADC_UPDT_RAW!r}")
    else:
        adc_updt = None
    return event_id, adc_updt


def key_filter(frame, event_id, adc_updt):
    return frame.filter(
        (F.col("run_id") == RUN_ID)
        & (F.col("EVENT_ID") == int(event_id))
        & F.col("ADC_UPDT").eqNullSafe(F.lit(adc_updt).cast("timestamp"))
    )


def key_sql_predicate(event_id, adc_updt):
    adc_sql = "NULL" if adc_updt is None else f"TIMESTAMP {sql_string(adc_updt)}"
    return (
        f"run_id = {sql_string(RUN_ID)} "
        f"AND EVENT_ID = {int(event_id)} "
        f"AND ADC_UPDT <=> {adc_sql}"
    )


def get_run():
    rows = spark.table(RUNS).filter(F.col("run_id") == RUN_ID).limit(2).collect()
    if len(rows) != 1:
        raise RuntimeError(f"Expected one run for {RUN_ID!r}; found {len(rows)}")
    return rows[0]


def require_review_fields():
    if not REVIEWER:
        raise ValueError("REVIEWER is required for this action")
    if not RESOLUTION:
        raise ValueError("RESOLUTION is required for this action")


def show_quarantine():
    frame = spark.table(QUARANTINE)
    if RUN_ID:
        frame = frame.filter(F.col("run_id") == RUN_ID)
    if EVENT_ID_RAW:
        frame = frame.filter(F.col("EVENT_ID") == int(EVENT_ID_RAW))
    if ADC_UPDT_RAW:
        parsed = spark.sql(
            f"SELECT try_cast({sql_string(ADC_UPDT_RAW)} AS TIMESTAMP) AS value"
        ).first()["value"]
        frame = frame.filter(F.col("ADC_UPDT").eqNullSafe(F.lit(parsed).cast("timestamp")))
    display(frame.orderBy(F.col("last_failure_ts").desc_nulls_last()))


def requeue_key():
    require_review_fields()
    event_id, adc_updt = parsed_event_key()
    meta = get_run()
    if meta["status"] in {
        "merging", "merge_failed", "merge_complete",
        "files_processing", "files_partial", "complete",
    }:
        raise RuntimeError(
            f"Run {RUN_ID} is already at {meta['status']}; create an explicit repair run "
            "instead of reopening a merged run."
        )

    matches = key_filter(spark.table(QUARANTINE), event_id, adc_updt)
    if matches.count() == 0:
        raise RuntimeError("No quarantine record matches the supplied key")

    completed = key_filter(
        completed_output_keys(BATCH, HISTORY, RUN_ID), event_id, adc_updt
    ).count()
    if completed:
        raise RuntimeError(
            "The key already has both batch and history output; resolve the "
            "completed/quarantined overlap before requeueing."
        )

    predicate = key_sql_predicate(event_id, adc_updt)
    spark.sql(f"""
      UPDATE {quote_table(QUARANTINE)}
      SET status = 'requeued',
          nonretryable_attempts = 0,
          resolution = {sql_string(RESOLUTION)},
          reviewed_by = {sql_string(REVIEWER)},
          reviewed_ts = current_timestamp(),
          last_failure_ts = current_timestamp()
      WHERE {predicate}
    """)

    if meta["status"] == "processing_complete":
        spark.sql(f"""
          UPDATE {quote_table(RUNS)}
          SET status = 'processing_partial',
              processing_completed_ts = NULL,
              error_message = NULL,
              updated_ts = current_timestamp()
          WHERE run_id = {sql_string(RUN_ID)}
        """)

    log_pipeline_event(
        EVENT_LOG,
        "WARN",
        "quarantine_key_requeued",
        f"Quarantined key requeued by {REVIEWER}",
        pipeline_id=meta["pipeline_id"],
        run_id=RUN_ID,
        event_id=event_id,
        adc_updt=adc_updt,
        details={"resolution": RESOLUTION},
    )
    print("Key requeued. Re-run Blob 2_Processing v4 for this RUN_ID.")


def waive_key():
    require_review_fields()
    event_id, adc_updt = parsed_event_key()
    meta = get_run()
    if meta["status"] in {
        "merging", "merge_failed", "merge_complete",
        "files_processing", "files_partial", "complete",
    }:
        raise RuntimeError(f"Cannot waive a key after merge has started: {meta['status']}")

    predicate = key_sql_predicate(event_id, adc_updt)
    matched = key_filter(spark.table(QUARANTINE), event_id, adc_updt).count()
    if matched == 0:
        raise RuntimeError("No quarantine record matches the supplied key")

    spark.sql(f"""
      UPDATE {quote_table(QUARANTINE)}
      SET status = 'waived',
          resolution = {sql_string(RESOLUTION)},
          reviewed_by = {sql_string(REVIEWER)},
          reviewed_ts = current_timestamp()
      WHERE {predicate}
    """)

    log_pipeline_event(
        EVENT_LOG,
        "WARN",
        "quarantine_key_waived",
        f"Quarantined key waived by {REVIEWER}",
        pipeline_id=meta["pipeline_id"],
        run_id=RUN_ID,
        event_id=event_id,
        adc_updt=adc_updt,
        details={"resolution": RESOLUTION},
    )
    print("Key waived. Re-run Blob 2_Processing v4 to recompute coverage.")


def retry_failed_shards():
    if not RUN_ID:
        raise ValueError("RUN_ID is required")
    meta = get_run()
    if meta["status"] not in {
        "worklist_ready", "processing", "processing_partial", "processing_failed"
    }:
        raise RuntimeError(f"Run {RUN_ID} cannot retry shards from status {meta['status']}")

    spark.sql(f"""
      UPDATE {quote_table(ATTEMPTS)}
      SET status = 'lease_expired',
          lease_owner = NULL,
          lease_expires_ts = NULL,
          heartbeat_ts = current_timestamp(),
          ended_ts = coalesce(ended_ts, current_timestamp())
      WHERE run_id = {sql_string(RUN_ID)}
        AND status IN ('submitted', 'queued', 'running')
        AND (lease_expires_ts IS NULL OR lease_expires_ts < current_timestamp())
    """)

    if meta["status"] == "processing_failed":
        spark.sql(f"""
          UPDATE {quote_table(RUNS)}
          SET status = 'processing_partial',
              error_message = NULL,
              lease_owner = NULL,
              lease_expires_ts = NULL,
              updated_ts = current_timestamp()
          WHERE run_id = {sql_string(RUN_ID)}
        """)

    log_pipeline_event(
        EVENT_LOG,
        "INFO",
        "failed_shards_reopened",
        "Failed or expired shards were reopened for idempotent retry",
        pipeline_id=meta["pipeline_id"],
        run_id=RUN_ID,
    )
    print("Run reopened. Invoke Blob 2_Processing v4 with this RUN_ID; completed keys are skipped.")


def release_stale_leases():
    spark.sql(f"""
      UPDATE {quote_table(RUNS)}
      SET lease_owner = NULL, lease_expires_ts = NULL, updated_ts = current_timestamp()
      WHERE lease_expires_ts IS NOT NULL AND lease_expires_ts < current_timestamp()
    """)
    spark.sql(f"""
      UPDATE {quote_table(ATTEMPTS)}
      SET status = CASE
            WHEN status IN ('submitted', 'queued', 'running') THEN 'lease_expired'
            ELSE status
          END,
          lease_owner = NULL,
          lease_expires_ts = NULL,
          ended_ts = CASE
            WHEN status IN ('submitted', 'queued', 'running') THEN coalesce(ended_ts, current_timestamp())
            ELSE ended_ts
          END
      WHERE lease_expires_ts IS NOT NULL AND lease_expires_ts < current_timestamp()
    """)
    spark.sql(f"""
      UPDATE {quote_table(FILE_QUEUE)}
      SET status = CASE WHEN status = 'processing' THEN 'retryable' ELSE status END,
          lease_owner = NULL,
          lease_expires_ts = NULL,
          updated_ts = current_timestamp()
      WHERE lease_expires_ts IS NOT NULL AND lease_expires_ts < current_timestamp()
    """)
    print("Expired run, shard-attempt, and file-queue leases released.")


def gc_candidate_runs():
    return (
        spark.table(RUNS)
        .filter(
            (F.col("status") == "complete")
            & (
                F.coalesce(
                    F.col("files_completed_ts"),
                    F.col("merged_ts"),
                    F.col("updated_ts"),
                )
                < F.current_timestamp() - F.expr(
                    f"INTERVAL {GC_RUN_RETENTION_DAYS} DAYS"
                )
            )
        )
        .select("run_id")
        .distinct()
    )


def gc_preview():
    inbox_candidates = (
        spark.table(INBOX).alias("i")
        .join(
            spark.table(CHECKPOINT).alias("c"),
            (F.col("i.pipeline_id") == F.col("c.pipeline_id"))
            & (F.col("i.source_table") == F.col("c.source_table"))
            & (F.col("i.trust_filter") == F.col("c.trust_filter"))
            & (F.col("i.commit_version") <= F.col("c.target_merged_version")),
            "inner",
        )
        .filter(
            F.col("i.ingested_ts")
            < F.current_timestamp() - F.expr(
                f"INTERVAL {GC_INBOX_RETENTION_DAYS} DAYS"
            )
        )
    )
    old_runs = gc_candidate_runs()
    summary = [
        ("cdf_inbox_rows", inbox_candidates.count()),
        ("completed_runs", old_runs.count()),
        ("run_chunk_rows", spark.table(RUN_CHUNKS).join(old_runs, "run_id", "inner").count()),
        ("batch_output_rows", spark.table(BATCH).join(old_runs, "run_id", "inner").count()),
        ("history_output_rows", spark.table(HISTORY).join(old_runs, "run_id", "inner").count()),
        ("file_queue_rows", spark.table(FILE_QUEUE).join(old_runs, "run_id", "inner").count()),
    ]
    display(spark.createDataFrame(summary, ["candidate_kind", "rows"]))
    print(
        "Policy: logically delete merged inbox rows after "
        f"{GC_INBOX_RETENTION_DAYS} days; delete durable run payloads after "
        f"{GC_RUN_RETENTION_DAYS} days; then VACUUM at 168 hours (7 days)."
    )


def gc_execute():
    if DRY_RUN:
        raise RuntimeError("Set DRY_RUN=false to execute logical cleanup and VACUUM")
    gc_preview()

    spark.sql(f"""
      DELETE FROM {quote_table(INBOX)} AS i
      WHERE i.ingested_ts < current_timestamp() - INTERVAL {GC_INBOX_RETENTION_DAYS} DAYS
        AND EXISTS (
          SELECT 1
          FROM {quote_table(CHECKPOINT)} AS c
          WHERE c.pipeline_id = i.pipeline_id
            AND c.source_table = i.source_table
            AND c.trust_filter = i.trust_filter
            AND c.target_merged_version IS NOT NULL
            AND i.commit_version <= c.target_merged_version
        )
    """)

    old_run_predicate = f"""
      run_id IN (
        SELECT run_id
        FROM {quote_table(RUNS)}
        WHERE status = 'complete'
          AND coalesce(files_completed_ts, merged_ts, updated_ts)
              < current_timestamp() - INTERVAL {GC_RUN_RETENTION_DAYS} DAYS
      )
    """
    for table_name in (
        FILE_QUEUE, HISTORY, BATCH, ATTEMPTS, QUARANTINE, RUN_CHUNKS, RUN_EVENTS
    ):
        spark.sql(f"DELETE FROM {quote_table(table_name)} WHERE {old_run_predicate}")

    for table_name in (
        INBOX, RUN_CHUNKS, RUN_EVENTS, BATCH, HISTORY,
        ATTEMPTS, QUARANTINE, FILE_QUEUE,
    ):
        spark.sql(f"VACUUM {quote_table(table_name)} RETAIN 168 HOURS")

    print("Blob v4 GC completed with the enforced 7-day physical-file retention.")


In [0]:
if ACTION == "status":
    display_status()
elif ACTION == "quarantine":
    show_quarantine()
elif ACTION == "requeue_key":
    requeue_key()
elif ACTION == "waive_key":
    waive_key()
elif ACTION == "retry_failed_shards":
    retry_failed_shards()
elif ACTION == "release_stale_leases":
    release_stale_leases()
elif ACTION == "gc_preview":
    gc_preview()
elif ACTION == "gc_execute":
    gc_execute()
else:
    raise ValueError(f"Unknown ACTION: {ACTION}")